# 18 -- High-gamma_ij runtime scaling

Numerical convergence of the locked production parameters (`N_min=128`,
`panels_per_oscillation=40`, `wall_points=20`, `how_often_ds=5`) at high
`gamma_ij` was already checked in `14_high_gamma_convergence_check.ipynb` --
not repeated here. This notebook is about **runtime**: now that field
evolution, `phi2` construction, and `precompute_z_kernel` all run on-device
(see recent `cpp/bubblemaster/src/gpu_integrator.cu` history), how does
wall-clock time actually scale with `gamma_ij`?

Runs `gamma_ij = 12, 16, 20, 24` at the locked production parameters, one
pair per GPU, all four concurrently on a single 4-GPU node. Each setup uses
its own pair's `R_*`/`T_MAX` (not a domain-shared value -- see
`14_high_gamma_convergence_check.ipynb`'s markdown for why that distinction
matters), since this is about a single pair's own numerics, not a scan.

**Workflow:** run the setup-building cells, submit the SLURM script, then
run the analysis section once it finishes.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import brentq

import ptbuilder as pt
from ptbuilder.ic import BubbleMasterParams, write_2d_setup
from ptbuilder.physics import Phi4Potential, PhysicsModel

ROOT = Path(pt.__file__).parent.parent
LAMBDA_BAR = 0.84

model = PhysicsModel(Phi4Potential(LAMBDA_BAR))

GAMMA_IJ_TARGETS = [12, 16, 20, 24]

KR_MIN, KR_MAX = 1.0, 100.0
N_OMEGA = 32
N_K     = 51
DT      = 2.0
TIME_FACTOR       = 1.4
PAIR_GAMMA_FACTOR = 2.5   # matches generate_surrogate's default; used only for
                          # omega_max's "smallest paired gamma" convention below

# Locked production values -- validated for convergence up to gamma_ij=8 in
# 14_high_gamma_convergence_check.ipynb; this notebook takes them as given.
N_MIN                  = 128
PANELS_PER_OSCILLATION = 40
WALL_POINTS            = 20
HOW_OFTEN_DS           = 5
BABY_STEPS             = 100
CUTOFF_TYPE            = 0

GPU_ACCOUNT      = 'm3166_g'
GPU_S_BATCH_SIZE = 128

## 1. Build one setup per gamma_ij, using each pair's own R_*/T_MAX

In [ ]:
kin = model.kinematics

def time_at_gamma(gamma):
    if gamma <= 1.0:
        return 0.0
    return brentq(lambda t: float(kin.Gamma(t)) - gamma, 0.0, 1.0e7)

def rstar_at_gamma(gamma):
    t = time_at_gamma(gamma)
    return 2.0 * float(kin.R(t, r='mid'))


SCAN_ROOT   = ROOT / 'data' / 'lb0.84_high_gamma_runtime_scaling'
setup_dir   = SCAN_ROOT / 'setups'
output_root = SCAN_ROOT / 'gpu'
setup_dir.mkdir(parents=True, exist_ok=True)
output_root.mkdir(parents=True, exist_ok=True)

params = BubbleMasterParams(
    dz=None, wall_points=WALL_POINTS, n_w=N_OMEGA, n_k=N_K,
    how_often_ds=HOW_OFTEN_DS, baby_steps=BABY_STEPS, cutoff_type=CUTOFF_TYPE,
    t_0_scal=1.0, collision_radius='mid',
)

manifest_rows = []   # (gamma_ij, name, setup_rel, n_t)
for gamma_ij in GAMMA_IJ_TARGETS:
    r_star    = rstar_at_gamma(gamma_ij)
    t_max     = TIME_FACTOR * r_star
    omega_min = KR_MIN / r_star

    t_first = time_at_gamma(gamma_ij)
    n_t     = int(np.ceil((t_max - t_first) / DT)) + 1
    times   = np.linspace(t_first, t_max, n_t)

    smallest_target = max(1.0, gamma_ij / PAIR_GAMMA_FACTOR)
    omega_max = KR_MAX / rstar_at_gamma(smallest_target)
    omega = np.geomspace(omega_min, omega_max, N_OMEGA)

    name = f'g{gamma_ij:g}'
    setup_path = setup_dir / f'{name}.h5'
    write_2d_setup(model, gamma_ij, times, setup_path, params, wlist=omega)
    manifest_rows.append((gamma_ij, name, setup_path.relative_to(ROOT), n_t))
    print(f'gamma_ij={gamma_ij:g}:  R_*={r_star:.3f}  t_first={t_first:.3f}  '
          f'T_MAX={t_max:.3f}  n_t={n_t}')

manifest = SCAN_ROOT / 'manifest.tsv'
manifest.write_text(
    'gamma_ij\tname\tsetup\tn_t\n' +
    ''.join(f'{g:.12g}\t{n}\t{s}\t{nt}\n' for g, n, s, nt in manifest_rows)
)

## 2. SLURM script -- one node, 4 GPUs, one gamma_ij per GPU, concurrent

Exactly as many tasks as GPUs on a node, so each pair gets its own GPU with
nothing else competing for it -- unlike a dense scan's `xargs -P` worker
pool, this just backgrounds all four `srun` calls and waits.

In [ ]:
GPU_WALLTIME = '04:00:00'   # generous -- actual scaling with the now-optimized
                            # binary at gamma_ij up to 24 is what we're measuring

def write_slurm_script(manifest_rows, gpu_walltime='04:00:00', gpu_account='m3166_g'):
    """Writes scripts/lb0.84_high_gamma_runtime_scaling_gpu.sh: one node, one
    task per row, all launched in the background and waited on -- there are
    exactly as many rows as GPUs on a node, so no worker-pool queueing is
    needed (contrast notebook 15/17's xargs -P pattern for dense scans)."""
    (ROOT / 'logs').mkdir(exist_ok=True)
    n_tasks = len(manifest_rows)
    relative_output = output_root.relative_to(ROOT)

    run_blocks = []
    for gamma_ij, name, setup_rel, n_t in manifest_rows:
        output = f'{relative_output}/{name}'
        run_blocks.append(
            f'mkdir -p "{output}"\n'
            f'/usr/bin/time --verbose srun --exclusive --exact -N 1 -n 1 -c 32 --gpus-per-task=1 --cpu-bind=cores bin/bubblemaster_gpu {setup_rel} {output}/ --param {N_MIN} --filon-panels-per-osc {PANELS_PER_OSCILLATION} --s-batch-size {GPU_S_BATCH_SIZE} &\n'
            f'pids+=("$!")'
        )
    runs = '\n'.join(run_blocks)

    script_path = ROOT / 'scripts' / 'lb0.84_high_gamma_runtime_scaling_gpu.sh'
    script_path.write_text(f'''#!/bin/bash
#SBATCH --job-name=bm_hg_scaling
#SBATCH --output=logs/bm_hg_scaling_%j.out
#SBATCH --constraint=gpu
#SBATCH --qos=regular
#SBATCH -A {gpu_account}
#SBATCH --nodes=1
#SBATCH --ntasks={n_tasks}
#SBATCH --ntasks-per-node={n_tasks}
#SBATCH --cpus-per-task=32
#SBATCH --gpus-per-task=1
#SBATCH --time={gpu_walltime}
set -euo pipefail
REPO_ROOT="${{SLURM_SUBMIT_DIR:-$PWD}}"
cd "$REPO_ROOT"
pids=()
{runs}
status=0
for pid in "${{pids[@]}}"; do
    wait "$pid" || status=1
done
exit "$status"
''')
    script_path.chmod(0o755)
    print(f'Written: {script_path.relative_to(ROOT)}  ({n_tasks} tasks, 1 node)')
    return script_path


script_path = write_slurm_script(manifest_rows, gpu_walltime=GPU_WALLTIME, gpu_account=GPU_ACCOUNT)
print(f'Submit: sbatch {script_path.relative_to(ROOT)}')

## 3. Runtime scaling -- run after the SLURM job finishes

Parses each row's own `/usr/bin/time --verbose` block from the shared SLURM
log (matched by the row's own setup path in "Command being timed:", not by
position -- the four tasks run concurrently so their blocks can interleave
in the log, but each block is self-identifying). Auto-picks the most recent
`bm_hg_scaling_*.out`; pass `LOG_PATH` explicitly to override.

In [ ]:
import re


def parse_runtime_log(log_path):
    """Maps gamma_ij -> wall-clock seconds, parsed from /usr/bin/time
    --verbose blocks in a SLURM log with concurrently-interleaved output.
    Splits on "Command being timed:" so each block is self-contained
    (robust to the 4 tasks' output interleaving at the line level -- each
    block still holds its own command AND its own Elapsed line). The quoted
    command is also de-wrapped by stripping internal whitespace before
    matching: /usr/bin/time wraps long command lines at a fixed column,
    sometimes splitting mid-token (e.g. a real case seen elsewhere in this
    project: "row_0029.h5" printed as "ro" + newline + spaces + "w_0029.h5")
    -- paths never contain meaningful whitespace, so this is safe.
    """
    text = Path(log_path).read_text()
    runtimes = {}
    for block in text.split('Command being timed:')[1:]:
        m_cmd = re.search(r'"(.*?)"', block, re.DOTALL)
        cmd = re.sub(r'\s+', '', m_cmd.group(1)) if m_cmd else ''
        m_name = re.search(r'setups/g([0-9.]+)\.h5', cmd)
        m_time = re.search(r'Elapsed\(wallclock\)time.*?:([\d:.]+)',
                           re.sub(r'\s+', '', block))
        if not (m_name and m_time):
            continue
        gamma_ij = float(m_name.group(1))
        parts = [float(p) for p in m_time.group(1).split(':')]
        secs = parts[0]*60 + parts[1] if len(parts) == 2 else parts[0]*3600 + parts[1]*60 + parts[2]
        runtimes[gamma_ij] = secs
    return runtimes


LOG_PATH = sorted((ROOT / 'logs').glob('bm_hg_scaling_*.out'))[-1]
print(f'Using log: {LOG_PATH.relative_to(ROOT)}')

runtimes = parse_runtime_log(LOG_PATH)
missing = [g for g in GAMMA_IJ_TARGETS if g not in runtimes]
if missing:
    print(f'Not found in log (job may still be running): {missing}')

gammas = np.array(sorted(runtimes))
secs   = np.array([runtimes[g] for g in gammas])
for g, s in zip(gammas, secs):
    print(f'  gamma_ij={g:g}:  {s:.1f} s  ({s/60:.1f} min)')

fig, ax = plt.subplots(figsize=(6, 5))
ax.loglog(gammas, secs, 'o-')
ax.set_xlabel('gamma_ij')
ax.set_ylabel('wall time [s]')
ax.set_title('High-gamma_ij runtime scaling (single pair, single GPU)')

if len(gammas) >= 2:
    p, logC = np.polyfit(np.log(gammas), np.log(secs), 1)
    gfit = np.geomspace(gammas.min(), gammas.max(), 50)
    ax.loglog(gfit, np.exp(logC) * gfit**p, '--', color='0.5',
             label=f'fit: time ~ gamma_ij^{p:.2f}')
    ax.legend(frameon=False)
    print(f'\nFitted scaling: time ~ gamma_ij^{p:.2f}')

plt.tight_layout()
plt.savefig(ROOT / 'high_gamma_runtime_scaling.pdf', bbox_inches='tight')
plt.show()